In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import shap
import joblib

df = pd.read_csv('../data/European_Bank.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/European_Bank.csv'

In [2]:
# 2. Data Cleaning (Handling duplicates and any hypothetical missing values)
df_cleaned = df.drop_duplicates()
for col in ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
for col in ['Geography', 'Gender']:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

In [3]:
# 3. Drop non-informative columns
columns_to_drop = ['Year', 'CustomerId', 'Surname']
df_cleaned = df_cleaned.drop(columns=[col for col in columns_to_drop if col in df_cleaned.columns])

In [4]:
# 4. Feature Engineering (Creating new predictive signals)
df_cleaned['Balance_Salary_Ratio'] = df_cleaned['Balance'] / df_cleaned['EstimatedSalary']
df_cleaned['Product_Density'] = df_cleaned['NumOfProducts'] / (df_cleaned['Tenure'] + 0.1)
df_cleaned['Engagement_Product_Interaction'] = df_cleaned['IsActiveMember'] * df_cleaned['NumOfProducts']
df_cleaned['Age_Tenure_Interaction'] = df_cleaned['Age'] * df_cleaned['Tenure']

In [5]:
# 5. One-Hot Encoding
df_encoded = pd.get_dummies(df_cleaned, columns=['Geography', 'Gender'], drop_first=True)
print("Data Preprocessing and Engineering Complete. Shape:", df_encoded.shape)

Data Preprocessing and Engineering Complete. Shape: (10000, 16)


In [6]:
# 1. Define Features and Target
X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']

In [7]:
# 2. Train-Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [8]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

print("Logistic Regression Results")

print(classification_report(y_test, lr_pred))

lr_auc = roc_auc_score(
    y_test,
    lr_model.predict_proba(X_test_scaled)[:,1]
)

print("ROC-AUC:", lr_auc)

Logistic Regression Results
              precision    recall  f1-score   support

           0       0.82      0.96      0.89      1593
           1       0.57      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.70      0.58      0.59      2000
weighted avg       0.77      0.81      0.77      2000

ROC-AUC: 0.7735640108521464


In [9]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Decision Tree Results")

print(classification_report(y_test, dt_pred))

dt_auc = roc_auc_score(
    y_test,
    dt_model.predict_proba(X_test)[:,1]
)

print("ROC-AUC:", dt_auc)

Decision Tree Results
              precision    recall  f1-score   support

           0       0.87      0.86      0.86      1593
           1       0.47      0.49      0.48       407

    accuracy                           0.79      2000
   macro avg       0.67      0.67      0.67      2000
weighted avg       0.79      0.79      0.79      2000

ROC-AUC: 0.6741911402928352


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Results")

print(classification_report(y_test, rf_pred))

rf_auc = roc_auc_score(
    y_test,
    rf_model.predict_proba(X_test)[:,1]
)

print("ROC-AUC:", rf_auc)

Random Forest Results
              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1593
           1       0.76      0.47      0.58       407

    accuracy                           0.86      2000
   macro avg       0.82      0.71      0.75      2000
weighted avg       0.85      0.86      0.85      2000

ROC-AUC: 0.8563648394156869


In [11]:
# 3. Train XGBoost Model
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

c:\Users\Ashmit\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:23:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [12]:
# 4. Evaluate Model
print("\n--- XGBoost Performance ---")
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test_scaled)[:, 1]):.4f}")
print(classification_report(y_test, xgb_model.predict(X_test_scaled)))


--- XGBoost Performance ---
ROC-AUC: 0.8307
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      1593
           1       0.68      0.48      0.57       407

    accuracy                           0.85      2000
   macro avg       0.78      0.71      0.74      2000
weighted avg       0.84      0.85      0.84      2000



In [13]:
# 5. Save the trained model and scaler for the web app
joblib.dump(xgb_model, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
# Save column names to ensure inputs match later
joblib.dump(list(X_train.columns), 'model_columns.pkl')
print("Model and Scaler saved successfully!")

Model and Scaler saved successfully!
